In [ ]:
! pip install tensorflow==2.21.0
! pip install matplotlib seaborn scikit-learn

In [ ]:
! kaggle datasets download lucaslucas1234/agricultural-crops-image-classification-split-20

Dataset URL: https://www.kaggle.com/datasets/lucaslucas1234/agricultural-crops-image-classification-split-20
License(s): MIT
agricultural-crops-image-classification-split-20.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)
from typing import Literal
import zipfile

keras.config.enable_unsafe_deserialization()

In [ ]:
print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponivel: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow: 2.21.0
GPU disponivel: True


In [ ]:
with zipfile.ZipFile("agricultural-crops-image-classification-split-20.zip", "r") as zip_ref:
  zip_ref.extractall("dataset")

In [ ]:
TRAIN_PATH = Path("dataset/dataset_split_20/train")
TEST_PATH = Path("dataset/dataset_split_20/test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42
EPOCHS = 20
LR = 1e-3

CLASS_NAMES: list[str] = sorted([p.name for p in TRAIN_PATH.iterdir() if p.is_dir()])
NUM_CLASSES: int = len(CLASS_NAMES)

AUTOTUNE = tf.data.AUTOTUNE

OUTPUT_DIR = Path("output")

DPI = 300

EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [ ]:
experiments = [
    {"model_name": "EfficientNetB0", "use_val": False, "use_aug": False},
    {"model_name": "EfficientNetB0", "use_val": True, "use_aug": False},
    {"model_name": "EfficientNetB0", "use_val": True, "use_aug": True},
    {"model_name": "MobileNetV2", "use_val": False, "use_aug": False},
    {"model_name": "ResNet50V2", "use_val": False, "use_aug": False},
    {"model_name": "DenseNet121", "use_val": False, "use_aug": False},
    {"model_name": "NASNetMobile", "use_val": False, "use_aug": False},
]

In [ ]:
def make_experiment_name(model_name: str, use_val: bool, use_aug: bool) -> str:
    val_part: Literal['com_val'] | Literal['sem_val'] = "com_val" if use_val else "sem_val"
    aug_part: Literal['com_aug'] | Literal['sem_aug'] = "com_aug" if use_aug else "sem_aug"
    return f"{model_name}_{val_part}_{aug_part}"

In [ ]:
def load_train_val_datasets(train_path: Path, use_val: bool):
    if use_val:
        VALIDATION_SPLIT = 0.20

        train_ds = tf.keras.utils.image_dataset_from_directory(
            train_path,
            validation_split=VALIDATION_SPLIT,
            subset="training",
            seed=SEED,
            image_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            class_names=CLASS_NAMES,
        )

        val_ds = tf.keras.utils.image_dataset_from_directory(
            train_path,
            validation_split=VALIDATION_SPLIT,
            subset="validation",
            seed=SEED,
            image_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            class_names=CLASS_NAMES,
        )

        return train_ds, val_ds

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_path,
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_names=CLASS_NAMES,
    )

    return train_ds, None

In [ ]:
def load_test_dataset(test_path: Path):
    return tf.keras.utils.image_dataset_from_directory(
        test_path,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        class_names=CLASS_NAMES,
    )

In [ ]:
def prepare_dataset(ds, shuffle: bool = False):
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED, reshuffle_each_iteration=True)
    return ds.cache().prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(15, 13))

for label, class_name in enumerate(CLASS_NAMES):
    class_dir: Path = TRAIN_PATH / class_name
    img_path: Path = next(p for p in class_dir.iterdir() if p.suffix.lower() in EXTS)

    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img = tf.keras.utils.img_to_array(img).astype("uint8")

    plt.subplot(5, 6, label + 1)
    plt.imshow(img)
    plt.title(class_name)
    plt.axis("off")

preview_dir: Path = OUTPUT_DIR / "dataset_preview"
preview_dir.mkdir(parents=True, exist_ok=True)

plt.tight_layout()
plt.savefig(preview_dir / "samples.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
def get_base_model(model_name: str):
    if model_name == "EfficientNetB0":
        return tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
        )
    if model_name == "MobileNetV2":
        return tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
        )
    if model_name == "ResNet50V2":
        return tf.keras.applications.ResNet50V2(
            include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
        )
    if model_name == "DenseNet121":
        return tf.keras.applications.DenseNet121(
            include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
        )
    if model_name == "NASNetMobile":
        return tf.keras.applications.NASNetMobile(
            include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
        )
    raise ValueError(f"Modelo não suportado: {model_name}")

In [ ]:
PREPROCESS_MODULES = {
    "EfficientNetB0": tf.keras.applications.efficientnet,
    "MobileNetV2": tf.keras.applications.mobilenet_v2,
    "ResNet50V2": tf.keras.applications.resnet_v2,
    "DenseNet121": tf.keras.applications.densenet,
    "NASNetMobile": tf.keras.applications.nasnet,
}

In [ ]:
def get_preprocess_fn(model_name: str):
    if model_name not in PREPROCESS_MODULES:
        raise ValueError(f"Preprocessamento não configurado para: {model_name}")

    return PREPROCESS_MODULES[model_name].preprocess_input

In [ ]:
def show_augmentations(
    train_path, aug_layers, output_path: Path, n_images=3, seed=42, img_size=IMG_SIZE
):
    tf.random.set_seed(seed)

    train_path = Path(train_path)

    image_paths = []

    for label, class_name in enumerate(CLASS_NAMES):
        class_dir: Path = train_path / class_name
        paths = sorted([p for p in class_dir.iterdir() if p.suffix.lower() in EXTS])

        if paths:
            image_paths.append((paths[0], label))

        if len(image_paths) >= n_images:
            break

    n_images = len(image_paths)
    n_cols = 1 + len(aug_layers)

    height_ratios = []
    for _ in range(n_images):
        height_ratios.extend([0.25, 3])

    fig = plt.figure(figsize=(3 * n_cols, 3.4 * n_images))

    gs = fig.add_gridspec(
        nrows=2 * n_images,
        ncols=n_cols,
        height_ratios=height_ratios,
        hspace=0.15,
        wspace=0.25,
    )

    for i, (img_path, label) in enumerate(image_paths):
        title_row = 2 * i
        image_row = 2 * i + 1

        ax_class = fig.add_subplot(gs[title_row, :])
        ax_class.axis("off")
        ax_class.text(
            0.5,
            0.5,
            CLASS_NAMES[label],
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            transform=ax_class.transAxes,
        )

        img = tf.keras.utils.load_img(img_path, target_size=img_size)
        img = tf.keras.utils.img_to_array(img)

        ax = fig.add_subplot(gs[image_row, 0])
        ax.imshow(img.astype("uint8"))
        ax.set_title("Original")
        ax.axis("off")

        img_tensor = tf.convert_to_tensor(img, dtype=tf.float32)

        for j, (name, layer) in enumerate(aug_layers.items(), start=1):
            aug_img = layer(tf.expand_dims(img_tensor, 0), training=True)[0]
            aug_img = tf.clip_by_value(aug_img, 0.0, 255.0)
            aug_img = tf.cast(aug_img, tf.uint8)

            ax = fig.add_subplot(gs[image_row, j])
            ax.imshow(aug_img.numpy())
            ax.set_title(name)
            ax.axis("off")

    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()

In [ ]:
def make_aug_layers():
    return {
        "Flip horizontal": layers.RandomFlip("horizontal"),
        "Rotação": layers.RandomRotation(0.1),
        "Zoom": layers.RandomZoom(0.1),
        "Contraste": layers.RandomContrast(0.2),
        "Translação": layers.RandomTranslation(0.05, 0.05),
    }

In [ ]:
preview_aug_layers = make_aug_layers()
show_augmentations(
    train_path=TRAIN_PATH,
    aug_layers=preview_aug_layers,
    output_path=preview_dir / "augmentation_preview.png",
    n_images=3,
)

In [ ]:
def make_data_augmentation():
    aug_layers = make_aug_layers()

    data_augmentation = keras.Sequential(
        [
            *aug_layers.values(),
        ],
        name="data_augmentation",
    )

    return data_augmentation

In [ ]:
def build_model(model_name: str, use_aug: bool):
    base_model = get_base_model(model_name)
    base_model.trainable = False

    preprocess_input = get_preprocess_fn(model_name)

    inputs = keras.Input(shape=IMG_SIZE + (3,))

    x = inputs

    if use_aug:
        augmentation = make_data_augmentation()
        x = augmentation(x)

    x = preprocess_input(x)

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs)

In [ ]:
def save_history_plot(history, output_path: Path, use_val: bool):
    history_dict = history.history
    epochs_range = range(1, len(history_dict["loss"]) + 1)

    plt.figure(figsize=(18, 5))

    # Acurácia
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, history_dict["accuracy"], label="Treino")

    if use_val and "val_accuracy" in history_dict:
        plt.plot(epochs_range, history_dict["val_accuracy"], label="Validação")

    plt.title("Acurácia")
    plt.xlabel("Época")
    plt.ylabel("Acurácia")
    plt.legend()

    # Top-3 Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, history_dict["top_3_accuracy"], label="Treino")

    if use_val and "val_top_3_accuracy" in history_dict:
        plt.plot(epochs_range, history_dict["val_top_3_accuracy"], label="Validação")

    plt.title("Top-3 Accuracy")
    plt.xlabel("Época")
    plt.ylabel("Top-3 Accuracy")
    plt.legend()

    # Loss
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, history_dict["loss"], label="Treino")

    if use_val and "val_loss" in history_dict:
        plt.plot(epochs_range, history_dict["val_loss"], label="Validação")

    plt.title("Loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()

In [ ]:
training_results = []

In [ ]:
for experiment in experiments:
    model_name = experiment["model_name"]
    use_val = experiment["use_val"]
    use_aug = experiment["use_aug"]

    experiment_name = make_experiment_name(model_name, use_val, use_aug)
    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 50)
    print(f"TREINAMENTO: {experiment_name}")
    print("=" * 50)

    train_ds, val_ds = load_train_val_datasets(TRAIN_PATH, use_val)

    train_ds = prepare_dataset(train_ds, shuffle=True)

    if use_val:
        val_ds = prepare_dataset(val_ds, shuffle=False)

    model = build_model(model_name, use_aug)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top_3_accuracy"),
        ],
    )

    monitor_metric: Literal["val_loss"] | Literal["loss"] = (
        "val_loss" if use_val else "loss"
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor=monitor_metric, patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor_metric, factor=0.3, patience=4, min_lr=1e-6, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=experiment_dir / "best_model.keras",
            monitor=monitor_metric,
            save_best_only=True,
        ),
        keras.callbacks.CSVLogger(
            filename=experiment_dir / "training_log.csv",
        ),
    ]

    start_time = time.perf_counter()
    if use_val:
        history = model.fit(
            train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks
        )
    else:
        history = model.fit(train_ds, epochs=EPOCHS, callbacks=callbacks)
    end_time = time.perf_counter()

    final_model_path = experiment_dir / "final_model.keras"
    model.save(final_model_path)

    with open(experiment_dir / "history.json", "w", encoding="utf-8") as f:
        json.dump(history.history, f, indent=4)

    save_history_plot(
        history=history,
        output_path=experiment_dir / "history.png",
        use_val=use_val,
    )

    training_info = {
        "experiment_name": experiment_name,
        "model_name": model_name,
        "use_val": use_val,
        "use_aug": use_aug,
        "training_duration_s": end_time - start_time,
        "final_model_path": str(final_model_path),
        "best_model_path": str(experiment_dir / "best_model.keras"),
        "output_dir": str(experiment_dir),
    }

    training_results.append(training_info)

    with open(experiment_dir / "training_info.json", "w", encoding="utf-8") as f:
        json.dump(training_info, f, indent=4)

TREINAMENTO: EfficientNetB0_sem_val_sem_aug
Found 652 files belonging to 30 classes.
Epoch 1/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 66s 655ms/step - accuracy: 0.2316 - loss: 2.9516 - top_3_accuracy: 0.3911 - learning_rate: 0.0010
Epoch 2/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.6288 - loss: 1.7304 - top_3_accuracy: 0.8512 - learning_rate: 0.0010
Epoch 3/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.7715 - loss: 1.1572 - top_3_accuracy: 0.9402 - learning_rate: 0.0010
Epoch 4/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.8497 - loss: 0.8622 - top_3_accuracy: 0.9678 - learning_rate: 0.0010
Epoch 5/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.8896 - loss: 0.6888 - top_3_accuracy: 0.9755 - learning_rate: 0.0010
TREINAMENTO: EfficientNetB0_com_val_sem_aug
Found 652 files belonging to 30 classes.
Using 522 files for training.
Found 652 files belonging to 30 classes.
Using 130 files for validation.
Epoch 1/5
33/33 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy:

In [ ]:
def predict_dataset(model, test_ds):
    y_true = []
    y_pred = []
    y_score = []

    for images, labels in test_ds:
        probs = model.predict(images, verbose=0)

        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(probs, axis=1))
        y_score.extend(probs)

    return np.array(y_true), np.array(y_pred), np.array(y_score)

In [ ]:
def save_confusion_matrices(y_true, y_pred, output_dir: Path):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = confusion_matrix(y_true, y_pred, normalize="true")

    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão")
    plt.tight_layout()
    plt.savefig(output_dir / "confusion_matrix.png", dpi=DPI, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão Normalizada")
    plt.tight_layout()
    plt.savefig(
        output_dir / "confusion_matrix_normalized.png", dpi=DPI, bbox_inches="tight"
    )
    plt.close()

    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cm_norm * 100,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão Normalizada (%)")
    plt.tight_layout()
    plt.savefig(
        output_dir / "confusion_matrix_percent.png", dpi=DPI, bbox_inches="tight"
    )
    plt.close()

In [ ]:
def compute_auc_metrics(y_true, y_score, num_classes):
    roc_auc_per_class = []
    pr_auc_per_class = []
    support_per_class = []

    for class_idx in range(num_classes):
        y_true_binary = (y_true == class_idx).astype(int)

        if len(np.unique(y_true_binary)) < 2:
            continue

        class_scores = y_score[:, class_idx]

        roc_auc = roc_auc_score(y_true_binary, class_scores)
        pr_auc = average_precision_score(y_true_binary, class_scores)

        support = int(y_true_binary.sum())

        roc_auc_per_class.append(roc_auc)
        pr_auc_per_class.append(pr_auc)
        support_per_class.append(support)

    if len(roc_auc_per_class) == 0:
        return {
            "macro_roc_auc_ovr": None,
            "weighted_roc_auc_ovr": None,
            "macro_pr_auc": None,
            "weighted_pr_auc": None,
        }

    roc_auc_per_class = np.array(roc_auc_per_class)
    pr_auc_per_class = np.array(pr_auc_per_class)
    support_per_class = np.array(support_per_class)

    return {
        "macro_roc_auc_ovr": float(np.mean(roc_auc_per_class)),
        "weighted_roc_auc_ovr": float(
            np.average(roc_auc_per_class, weights=support_per_class)
        ),
        "macro_pr_auc": float(np.mean(pr_auc_per_class)),
        "weighted_pr_auc": float(
            np.average(pr_auc_per_class, weights=support_per_class)
        ),
    }

In [ ]:
evaluation_results = []

In [ ]:
test_ds = load_test_dataset(TEST_PATH)
test_ds = prepare_dataset(test_ds, shuffle=False)

Found 177 files belonging to 30 classes.


In [ ]:
for result in training_results:
    experiment_name = result["experiment_name"]
    output_dir = Path(result["output_dir"])

    print("=" * 50)
    print(f"AVALIAÇÃO: {experiment_name}")
    print("=" * 50)

    model_path = result["best_model_path"]
    model = tf.keras.models.load_model(model_path)

    test_loss, test_acc, test_top3_acc = model.evaluate(test_ds, verbose=0)

    y_true, y_pred, y_score = predict_dataset(model, test_ds)

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
        output_dict=True,
    )

    report_text = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    auc_metrics = compute_auc_metrics(
        y_true=y_true,
        y_score=y_score,
        num_classes=NUM_CLASSES,
    )

    with open(output_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    with open(output_dir / "classification_report.json", "w", encoding="utf-8") as f:
        json.dump(report_dict, f, indent=4)

    save_confusion_matrices(y_true, y_pred, output_dir)

    metrics = {
        "experiment_name": experiment_name,
        "model_name": result["model_name"],
        "use_val": result["use_val"],
        "use_data_augmentation": result["use_aug"],
        "training_duration_s": result["training_duration_s"],
        "test_loss": float(test_loss),
        "test_acc": float(test_acc),
        "test_top3_acc": float(test_top3_acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        **auc_metrics,
        "final_model_path": result["final_model_path"],
        "best_model_path": result["best_model_path"],
        "output_dir": result["output_dir"],
    }

    evaluation_results.append(metrics)

    with open(output_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=4)

In [ ]:
def save_comparison_plots(evaluation_results, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(evaluation_results)
    df.to_csv(output_dir / "evaluation_results.csv", index=False)

    df["label"] = df["experiment_name"]

    metrics = [
        ("test_acc", "Acurácia no Teste"),
        ("test_top3_acc", "Top-3 Accuracy no Teste"),
        ("test_loss", "Loss no Teste"),
        ("macro_f1", "Macro F1-score"),
        ("weighted_f1", "Weighted F1-score"),
        ("macro_precision", "Macro Precision"),
        ("macro_recall", "Macro Recall"),
        ("macro_roc_auc_ovr", "Macro ROC-AUC One-vs-Rest"),
        ("weighted_roc_auc_ovr", "Weighted ROC-AUC One-vs-Rest"),
        ("macro_pr_auc", "Macro PR-AUC"),
        ("weighted_pr_auc", "Weighted PR-AUC"),
    ]

    for metric, title in metrics:
        if metric not in df.columns:
            continue

        if df[metric].isna().all():
            continue

        plt.figure(figsize=(12, 6))

        ax = sns.barplot(data=df, x="label", y=metric)

        for container in ax.containers:
            ax.bar_label(container, fmt="%.4f", padding=3, fontsize=9)

        y_min = df[metric].min()
        y_max = df[metric].max()
        margin = (y_max - y_min) * 0.15 if y_max != y_min else 0.05

        ax.set_ylim(y_min * 0.98, y_max + margin)

        plt.title(title)
        plt.xlabel("Experimento")
        plt.ylabel(metric)

        plt.xticks(rotation=45, ha="right")

        plt.tight_layout()

        plt.savefig(
            output_dir / f"comparison_{metric}.png", dpi=DPI, bbox_inches="tight"
        )

        plt.close()

    return df

In [ ]:
comparison_dir = OUTPUT_DIR / "comparison"
df_results = save_comparison_plots(evaluation_results, comparison_dir)

In [ ]:
# Seleciona automaticamente o melhor experimento usando comparação hierárquica de métricas.
#
# Ordem de prioridade:
# 1) maior macro_f1
# 2) maior macro_recall
# 3) maior test_acc
# 4) maior test_top3_acc
# 5) menor test_loss
#
# A função max() compara as métricas da esquerda para a direita.
# O sinal negativo em test_loss é usado porque queremos minimizar a loss, enquanto max() sempre procura o maior valor.

best_result = max(
    evaluation_results,
    key=lambda r: (
        r["macro_f1"],
        r["macro_recall"],
        r["test_acc"],
        r["test_top3_acc"],
        -r["test_loss"],
    ),
)

print("Melhor modelo avaliado:\n")
print(f"Experimento: {best_result['experiment_name']}")
print(f"Macro F1:    {best_result['macro_f1']:.4f}")
print(f"Accuracy:    {best_result['test_acc']:.4f}")
print(f"Top-3 Acc:   {best_result['test_top3_acc']:.4f}")
print(f"Loss:        {best_result['test_loss']:.4f}")

best_model = tf.keras.models.load_model(best_result["best_model_path"])

Melhor modelo avaliado:

Experimento: EfficientNetB0_sem_val_sem_aug
Macro F1:    0.7847
Accuracy:    0.7910
Top-3 Acc:   0.9435
Loss:        0.9936


In [ ]:
def save_prediction_grid(
    model,
    test_ds,
    output_path: Path,
    n_batches: int = 3,
    n_images: int = 25,
    show: bool = False,
):
    """
    Gera uma grade com imagens de teste, rótulo real e predição do modelo.

    Por padrão, salva o gráfico e não exibe no notebook.
    Para exibir apenas o melhor modelo no final, use show=True somente nessa chamada.
    """
    idx_to_class = {i: name for i, name in enumerate(CLASS_NAMES)}

    test_ds_viz = test_ds.shuffle(buffer_size=1000, seed=SEED)

    all_images = []
    all_labels = []

    for x_batch, y_batch in test_ds_viz.take(n_batches):
        all_images.append(x_batch.numpy())
        all_labels.append(y_batch.numpy())

    x_test_viz = np.concatenate(all_images, axis=0)
    y_test_viz = np.concatenate(all_labels, axis=0)

    predictions = model.predict(x_test_viz, verbose=0)

    n_images = min(n_images, len(x_test_viz))

    plt.figure(figsize=(15, 15))

    for i in range(n_images):
        plt.subplot(5, 5, i + 1)

        img = x_test_viz[i]

        if img.max() <= 1.0:
            plt.imshow(img)
        else:
            plt.imshow(img.astype("uint8"))

        true_class_idx = int(y_test_viz[i])
        predicted_class_idx = int(np.argmax(predictions[i]))

        true_class_name = idx_to_class.get(true_class_idx, f"({true_class_idx})")
        predicted_class_name = idx_to_class.get(
            predicted_class_idx, f"({predicted_class_idx})"
        )

        color = "green" if true_class_idx == predicted_class_idx else "red"

        plt.title(
            f"Real: {true_class_name}\nPred: {predicted_class_name}",
            color=color,
            fontsize=9,
        )

        plt.axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

In [ ]:
best_output_dir = Path(best_result["output_dir"])

save_prediction_grid(
    model=best_model,
    test_ds=test_ds,
    output_path=best_output_dir / "predictions_grid_best_model.png",
    show=False,
)